In [1]:
import sys,os
sys.path.append(r'Z:/EnergyTrading/Python/')

from datetime import datetime, time, date
from Database.TPData import TPData, TPDataDa
from SynthSpread.spreadviewer_class import SpreadSingle
import pandas as pd
import matplotlib.pyplot as plt
from Strategies.Arbitrage_strategy.backtest_class import BacktestArb


def group_trades(df_tr, agg_dict):
    # Group trades
    df_tr['count'] = 1
    df_tr['price'] *= df_tr['volume']
    df_tr = df_tr.groupby(df_tr.index).agg(agg_dict)
    df_tr['price'] /= df_tr['volume']
    return df_tr.loc[:, ['price', 'volume', 'action', 'broker_id']]


n_s = 0
mkt_list = ['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'fr', 'fr', 'fr', 'fr', 'fr', 'fr']
tenor_list = ['w', 'w', 'm', 'm', 'm', 'q', 'q', 'q', 'q', 'y', 'y', 'm', 'm', 'q', 'q', 'y', 'y']
tn1_list = [1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 1, 2, 1, 2, 1, 2]
# mkt_list = ['de']
# tenor_list = ['y']
# tn1_list = [1]
# mkt_list = ['it', 'it', 'it', 'es', 'es', 'es', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'fr', 'fr', 'fr', 'fr', 'fr', 'fr']
# tenor_list = ['m', 'q', 'y', 'm', 'q', 'y', 'm', 'm', 'q', 'q', 'q', 'q', 'y', 'y', 'm', 'q', 'q', 'y', 'y']
# tn1_list = [1, 1, 1, 1, 1, 1, 3, 4, 5, 6, 7, 8, 3, 4, 3, 3, 4, 3, 4]
# mkt_list = ['de']
# tenor_list = ['y']
# tn1_list = [1]
# mkt_list = ['de_fr', 'de_fr', 'de_fr', 'de_fr', 'de_fr', 'de_fr', 'de_fr', 'de_fr', 'de_fr']
# tenor_list = ['m', 'm', 'm','q', 'q', 'q', 'q', 'y', 'y']
# tn1_list = [1, 2, 3, 1, 2, 3, 4, 1, 2]
tn2_list = []
prod = 'base'
venue_list = ['eex']
start_date = date.today() 
end_date = date.today() 

if not tn2_list:
    tn_list = [str(t1) for t1 in tn1_list]
else:
    tn_list = [str(t1) + '_' + str(t2) for (t1, t2) in zip(tn1_list, tn2_list)]

dates = pd.date_range(start_date, end_date, freq='B')

spread_class = SpreadSingle(mkt_list, tenor_list, tn1_list, tn2_list, venue_list)
product_date1 = spread_class.product_dates(dates, n_s, tn_bool=True)
product_date2 = spread_class.product_dates(dates, n_s, tn_bool=False)

start_time = time(8, 0, 0, 0)
end_time = time(17, 55, 0, 0)

data_class = TPData()

is_ob = False
brk_ids = [1441]
agg_dict = {'price': 'first', 'volume': 'first', 'action': 'first',
            'broker_id': 'first', 'count': 'first'}

backtest_class = BacktestArb()

tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
mr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date1):
    df_tr = pd.DataFrame([])
    mrg_ser = pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # Trades
        data_class.create_connection('OracleSQL')
        try:
            df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
        except AttributeError:
            continue
        # Group trades
        df_tr_aux = group_trades(df_tr_aux, agg_dict).between_time(start_time, end_time)
        if df_tr.empty:
            df_tr = df_tr_aux
        else:
            df_tr = pd.concat([df_tr, df_tr_aux])
        # Orders
        df_ba = None
        try:
            mrg_ser_aux = backtest_class.trd_arb(df_ba, df_tr, brk_ids, side='all')
        except:
            mrg_ser_aux = pd.DataFrame([])
        if mrg_ser.empty:
            mrg_ser = mrg_ser_aux
        else:
            mrg_ser = pd.concat([mrg_ser, mrg_ser_aux])
    tr_data_dict[m + t + str(n)] = df_tr
    mr_data_dict[m + t + str(n)] = mrg_ser

opp_dict = {k: v[v != 0].dropna() for k, v in mr_data_dict.items()}

opp_dict_ = {}
stat_dict = {}
agg_dict_ = {'Index': 'first', 'mrg': 'first', 'time_diff': 'first', 'broker_id': 'first'}
for m, df in opp_dict.items():
    if df.empty:
        opp_dict_[m] = []
        stat_dict[m] = []
        continue
    df = df.sort_values(by='Index').reset_index()
    # Initialize indexing list
    indexing = [0]

    # Iterate through the dataframe to calculate the new index
    for i in range(1, len(df)):
        prev_time = df.loc[i - 1, 'Index'] + pd.to_timedelta(df.loc[i - 1, 'time_diff'], unit='s')
        current_time = df.loc[i, 'Index']

        # If the previous timestamp + time_diff is less than the current timestamp, increment the index
        if prev_time < current_time:
            indexing.append(indexing[-1] + 1)
        else:
            indexing.append(indexing[-1])

    # Add the new indexing to the dataframe
    df['new_index'] = indexing
    # df = df.set_index('Index')
    df_ = df.loc[df['mrg'] >= 0.07 - 0.001, :]
    df['mrg'] = df['mrg'].round(2)
    opp_dict_[m] = df_.groupby('new_index').agg(agg_dict_).set_index('Index')
    opp_dict_[m]['mrg'] = opp_dict_[m]['mrg'].round(2)
    stat_dict[m] = opp_dict_[m].groupby('mrg')['time_diff'].describe()

Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connec

In [2]:
result=[]
for key in opp_dict.keys():
    df_aux=opp_dict[key].reset_index().rename(columns={'Index': 'datetime'})
    df_aux['contract']=key
    #df_aux['broker_id']=df_aux['broker_id'].replace(6, 4)
    result.append(df_aux)

df=pd.concat(result)
df['broker_id']=df['broker_id'].replace(6, 4)
df['contract'].value_counts()

contract
dem1    166
dem2     60
frm1     55
frm2     46
deq1     28
dey1     24
dem3     17
deq2     15
dew1     11
deq3     11
dew2      8
frq1      7
dey2      5
Name: count, dtype: int64

In [3]:
from Database.DB_reader import Database

local_db_config_path=r'Z:\EnergyTrading\configDB.json'
conn = Database(path_name=local_db_config_path)
conn._connect()
conn.execute_general_query(f"delete from algo.test_mrg_trd_opp_dict where datetime>='{date.today()}';")
conn._disconnect()

conn._connect()
df.to_sql('test_mrg_trd_opp_dict', conn.engine, schema='algo', if_exists='append', index=False)
conn._disconnect()


Connected to the database postgre
Connected to the database postgre
Query Error: This result object does not return rows. It has been closed automatically.
Disconnected from the database postgre
Connected to the database postgre


453

In [4]:
result=[]
for key in opp_dict.keys():
    try:
        df_aux=opp_dict_[key].reset_index().rename(columns={'Index': 'datetime'})
        df_aux['contract']=key
        #df_aux['broker_id']=df_aux['broker_id'].replace(6, 4)
        result.append(df_aux)
    except:
        pass

df=pd.concat(result)
df['broker_id']=df['broker_id'].replace(6, 4)
df['contract'].value_counts()

contract
dem1    49
frm2    37
dem2    24
frm1    21
dem3    12
deq1    10
dey1     9
deq2     8
deq3     7
frq1     5
dew1     4
dew2     2
dey2     2
Name: count, dtype: int64

In [5]:
from Database.DB_reader import Database

local_db_config_path=r'Z:\EnergyTrading\configDB.json'
conn = Database(path_name=local_db_config_path)
conn._connect()
conn.execute_general_query(f"delete from algo.test_mrg_trd_opp_dict_ where datetime>='{date.today()}';")
conn._disconnect()

conn._connect()
df.to_sql('test_mrg_trd_opp_dict_', conn.engine, schema='algo', if_exists='append', index=False)
conn._disconnect()

Connected to the database postgre
Connected to the database postgre
Query Error: This result object does not return rows. It has been closed automatically.
Disconnected from the database postgre
Connected to the database postgre


190